# Day 5 — ILT 2: Data Quality Constraints & Validation

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Reads from** | `<your-catalog>.bronze.*` |
| **Feeds into** | Day 5 HOL 1 — Transformations + DQ + Quarantine |
| **Duration** | 90 minutes |
| **Catalog** | Your catalog (Unity Catalog) — code below shows GlobalMart's real run |

### Learning Objectives
- Write a DQ scan that tags every row with its first failing rule
- Use Delta Lake `CHECK` and `NOT NULL` constraints to enforce quality at write time
- Apply the quarantine-table pattern for rows that fail validation
- Learn the hardest DQ skill: knowing when a defect should be **fixed**, **flagged**, or **quarantined** — and why over-quarantining is its own kind of data quality bug

## The DQ Scan Pattern

The pattern used across every GlobalMart Silver notebook is the same: build one `_dq_issue` column with a chain of `when()` conditions, checked in priority order — the **first** matching rule wins. This gives a complete picture of every problem in the table in a single pass, before any fix is applied.

### Real example — the `bronze.customers` DQ scan

In [ ]:
from pyspark.sql.functions import col, lit, when, trim, floor, datediff, to_date

bronze_customers = spark.table("gbmart.bronze.customers")
EMAIL_REGEX = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'

dq_scan_df = bronze_customers \
    .withColumn("_dob_temp",   to_date(col("DateOfBirth"),      "yyyy-MM-dd")) \
    .withColumn("_reg_temp",   to_date(col("RegistrationDate"), "yyyy-MM-dd")) \
    .withColumn("_age_at_reg", floor(datediff(col("_reg_temp"), col("_dob_temp")) / 365.25)) \
    .withColumn("_dq_issue",
        # Order matters — this is a priority chain, not independent checks.
        # The FIRST matching condition is the one recorded; a row with two
        # problems still gets exactly one tag (the highest-priority one).
        when(col("CustomerID").isNull(),                                lit("NULL_CUSTOMER_ID"))
        .when(col("FirstName").isNull() | (trim(col("FirstName")) == ""), lit("NULL_FIRST_NAME"))
        .when(col("Email").isNull(),                                    lit("NULL_EMAIL"))
        .when(~col("Email").rlike(EMAIL_REGEX),                         lit("INVALID_EMAIL_FORMAT"))
        .when(col("_age_at_reg") < 18,                                  lit("REGISTERED_UNDER_18"))
        .otherwise(lit(None))
    )

print("=== DQ Issues Found ===")
dq_scan_df.groupBy("_dq_issue").count().orderBy("count", ascending=False).show()

---
## The Real Decision Isn't "Is This Dirty?" — It's "What Do I Do About It?"

Every flagged row needs one of three outcomes, and picking the wrong one is a quality bug in itself:

| Outcome | When to use it | GlobalMart example |
|---|---|---|
| **Fix** | The correct value is knowable or recoverable | Email with a stray space — remove the space, the customer is real |
| **Flag** | The row is usable, but a specific field is suspect — don't hide it, don't discard it | Delivery date appearing before shipping date (see below) |
| **Quarantine** | The row genuinely violates a business rule and shouldn't reach Silver | Customer registered under 18 (GlobalMart's ToS requires 18+) |

### Case study — REGISTERED_UNDER_18 (a clean quarantine decision)

77 customers in `bronze.customers` show an age-at-registration under 18. GlobalMart requires 18+ to register — legal compliance and payment authorization. Unlike the email-space issue, **we cannot assume which field is wrong** — the DOB could be a typo, or the customer really was underage when the (unvalidated, at the time) registration form let them through. Since we can't safely fix it, and letting it into Silver as a valid customer would be actively wrong, this is a quarantine. Silver will not include these 77 records; the business team decides next steps.

### Case study — DELIVERY_BEFORE_SHIP (why over-quarantining is its own bug)

This is the most important story in this notebook. `bronze.orders` has an early DQ scan that flags 725 orders where `actualdeliverydate < shippingdate` — physically impossible. Before fixing anything, look closer:

In [ ]:
from pyspark.sql.types import DateType
from pyspark.sql.functions import datediff

bronze_orders = spark.table("gbmart.bronze.orders")

# First pass compared raw TIMESTAMPS -> 725 flagged. Re-run at the DATE level:
date_level_flag = bronze_orders.withColumn(
    "_delivery_before_ship",
    col("actualdeliverydate").isNotNull() & col("shippingdate").isNotNull() &
    (col("actualdeliverydate").cast(DateType()) < col("shippingdate").cast(DateType()))
)

flagged = date_level_flag.filter(col("_delivery_before_ship"))
print(f"Flagged at date level: {flagged.count():,}  (was 725 at timestamp level)")

# Check how consistent the gap is -- random noise vs. a systemic pattern
flagged.select(
    datediff(col("shippingdate").cast(DateType()), col("actualdeliverydate").cast(DateType())).alias("days_diff")
).groupBy("days_diff").count().orderBy("days_diff", ascending=False).show()

**What the investigation found:** at the date level only 114 records remain (not 725 — most of the timestamp-level flags were a comparison artifact, not real errors), and every one of those 114 shows exactly a **1-day** difference. That consistency points to a systemic cause: Supabase stores timestamps in UTC, GlobalMart's delivery scanners record in IST (UTC+5:30). Near midnight, the same real-world moment can land on different calendar dates once cast down to a plain date:

```
Delivered : June 14, 23:45 IST  ->  stored as June 14
Shipped   : June 15, 00:30 IST  ->  stored as June 15
```

Physically the order shipped before it was delivered — only the date *label* flips.

**What happened when this was first quarantined:** an earlier version of the Silver pipeline quarantined all 114 orders outright. While building `silver.order_items` downstream, that decision turned out to orphan **352 line items** — their parent order no longer existed in `silver.orders`, silently hiding real revenue and quantity from every Gold report. The defect was confined to *two date columns*; `customer_id`, `order_channel`, and the line items themselves were never in question. Quarantining the entire order discarded far more than the actual defect justified — and the cost only showed up two tables later, which is exactly what makes over-quarantining dangerous: **it fails silently, downstream, somewhere else.**

**The corrected decision:** keep all orders, and add a `_data_note = 'POSSIBLE_TIMEZONE_OFFSET_1DAY'` column on the 114 affected rows. They stay fully joinable for revenue/quantity, and analysts can still filter them in or out for delivery-SLA analysis specifically.

---
## Enforcing Quality at Write Time — Delta Constraints

A DQ scan is a *check you remember to run*. A Delta constraint is enforced by the table itself — nobody can write a violating row, ever, even by accident, even from a different notebook next year.

| Constraint type | Syntax | GlobalMart example |
|---|---|---|
| `NOT NULL` | `ALTER TABLE t ALTER COLUMN c SET NOT NULL` | `customer_id`, `order_id`, `product_id` — every primary/foreign key |
| `CHECK` | `ALTER TABLE t ADD CONSTRAINT name CHECK (condition)` | `actual_price_inr > 0`, `rating BETWEEN 0 AND 5` |

### Real example — from `silver.products` DQ rules

The products DQ scan checks: `NULL_PRODUCT_ID`, `INVALID_PRICE` (price null or &lt;= 0), `DISCOUNT_EXCEEDS_PRICE` (discounted &gt; actual), `INVALID_RATING` (not in [0,5]). These are exactly the kind of rules that belong as table constraints once Silver is stable — turning a scan you have to remember into a guarantee the table enforces on its own.

In [ ]:
# Delta constraints on gbmart.silver.products (illustrative — run once Silver exists)
spark.sql("""
  ALTER TABLE gbmart.silver.products
  ALTER COLUMN product_id SET NOT NULL
""")

spark.sql("""
  ALTER TABLE gbmart.silver.products
  ADD CONSTRAINT valid_price CHECK (actual_price_inr > 0)
""")

spark.sql("""
  ALTER TABLE gbmart.silver.products
  ADD CONSTRAINT valid_rating CHECK (rating BETWEEN 0 AND 5)
""")

print("Constraints added — any future INSERT/MERGE violating these fails immediately, table-wide.")

> **Constraints are a backstop, not a replacement for the DQ scan.** The scan runs *before* the write and lets you route bad rows to quarantine gracefully. A constraint runs *at* the write and just fails the whole operation if violated — you want bad rows filtered out via the scan long before they'd ever hit a constraint.

---
## The Quarantine Table Pattern

Every table that needs one follows the same shape: split the DQ-scanned DataFrame into `clean_df` (no issue) and `quarantine_df` (issue present), write both — `silver.<table>` and `silver.<table>_quarantine`. Nothing is ever silently dropped; a rejected row is always findable somewhere.

In [ ]:
# Same dq_scan_df from the customers example above
clean_df = dq_scan_df.filter(col("_dq_issue").isNull()) \
                     .drop("_dq_issue", "_dob_temp", "_reg_temp", "_age_at_reg")

quarantine_df = dq_scan_df.filter(col("_dq_issue").isNotNull()) \
                          .drop("_dob_temp", "_reg_temp", "_age_at_reg")

print(f"Total rows  : {bronze_customers.count():,}")
print(f"Clean rows  : {clean_df.count():,}")
print(f"Quarantine  : {quarantine_df.count():,}")

# Quarantine breakdown -- lets the business team see WHY each row was rejected
quarantine_df.groupBy("_dq_issue").count().orderBy("count", ascending=False).show()

---
## Decision Checklist — Fix, Flag, or Quarantine?

Ask these in order:

1. **Is the correct value knowable or recoverable from elsewhere in the row?** (e.g. PinCode duplicated inside `AddressLine1`) -> **Fix**.
2. **Is the row still usable, but I shouldn't hide that one field is suspect?** -> **Flag** with a `_data_note`-style column; keep the row.
3. **Does keeping the row violate an actual business rule, and I can't safely guess the correct value?** -> **Quarantine**.
4. **Before quarantining anything, ask: what depends on this row downstream?** The `DELIVERY_BEFORE_SHIP` story exists because step 4 was skipped the first time.

Next: Day 5 HOL 1 — you'll apply this exact scan -> fix/flag/quarantine -> constraint pipeline yourself, then in HOL 2 build the full Silver layer across all 4 sources.